<a href="https://colab.research.google.com/github/Airplane356/video-background-remover/blob/main/SAM3_YOLO_Video_Background_Remover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Video Background Remover using Meta Segment Anything Model 3 (SAM3)</h1>

**Install**

In [ ]:
!git clone https://github.com/facebookresearch/sam3.git
%cd sam3
!pip install -e ".[notebooks]" -q
!pip install ultralytics -q

**Imports and authentication**

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from sam3.model_builder import build_sam3_video_predictor
from ultralytics import YOLO

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Imports OK. Device:", device)


**Config**

In [ ]:
INPUT_VIDEO  = "/content/video.mp4"
OUTPUT_VIDEO = "/content/output_nobg.mp4"
TEMP_VIDEO   = "/content/temp_sam3.mp4"

BG_COLOR = (0, 177, 64)   # BGR. (0,177,64)=green screen, (0,0,0)=black
MAX_DIM  = 1024            # SAM3 internal resolution cap

YOLO_WEIGHTS = "yolov8m.pt"  # yolov8n=fast / yolov8m=balanced / yolov8x=accurate

# Objects within this depth range of the closest detected object are foreground.
# 0.0 = only the single closest object. 1.0 = everything.
DEPTH_TOLERANCE = 0.30
CONF_THRESHOLD  = 0.30    # minimum YOLO confidence to consider


**Load video + rewrite working copy size**

In [ ]:
cap      = cv2.VideoCapture(INPUT_VIDEO)
W        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps      = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

scale = min(MAX_DIM / max(W, H), 1.0)
SW    = int(round(W * scale / 2)) * 2
SH    = int(round(H * scale / 2)) * 2
print(f"Original {W}x{H}  ->  SAM3 {SW}x{SH},  {n_frames} frames @ {fps:.2f} fps")

out_tmp = cv2.VideoWriter(TEMP_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (SW, SH))
pbar    = tqdm(total=n_frames, desc="Preparing temp video")
while True:
    ret, frame = cap.read()
    if not ret:
        break
    out_tmp.write(cv2.resize(frame, (SW, SH)))
    pbar.update(1)
cap.release()
out_tmp.release()
pbar.close()
print("Temp video ready.")

cap         = cv2.VideoCapture(TEMP_VIDEO)
ret, frame0 = cap.read()
cap.release()
assert ret, "Could not read frame 0!"
frame0_rgb = cv2.cvtColor(frame0, cv2.COLOR_BGR2RGB)


**YOLO detection on frame 0**

In [ ]:
# ── MiDaS depth estimation ───────────────────────────────────────────────────
print("Loading MiDaS depth estimator...")
try:
    midas    = torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid", pretrained=True, verbose=False)
    midas.to(device).eval()
    midas_tf = torch.hub.load("intel-isl/MiDaS", "transforms", verbose=False).dpt_transform

    with torch.no_grad():
        depth_raw = midas(midas_tf(frame0_rgb).to(device))
        depth_map = F.interpolate(
            depth_raw.unsqueeze(1), size=(SH, SW), mode="bicubic", align_corners=False
        ).squeeze().cpu().numpy()

    # MiDaS returns inverse depth: higher value = closer to camera
    depth_norm = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min() + 1e-8)
    USE_DEPTH  = True
    print("Depth map ready.")
except Exception as e:
    print(f"MiDaS unavailable ({e}). Falling back to confidence x area scoring.")
    depth_norm = None
    USE_DEPTH  = False

# ── YOLO detection ───────────────────────────────────────────────────────────
print("\nRunning YOLO...")
yolo      = YOLO(YOLO_WEIGHTS)
det       = yolo(frame0_rgb, verbose=False)[0]
yolo_dets = det.boxes

if len(yolo_dets) == 0:
    print("WARNING: YOLO found nothing.")
    fg_detections = []
else:
    confs = yolo_dets.conf.cpu().numpy()
    xyxy  = yolo_dets.xyxy.cpu().numpy().astype(int)
    clses = yolo_dets.cls.cpu().numpy().astype(int)

    if USE_DEPTH:
        det_scores = []
        for i in range(len(yolo_dets)):
            x1, y1, x2, y2 = np.clip(xyxy[i], [0,0,0,0], [SW-1,SH-1,SW-1,SH-1])
            region = depth_norm[y1:y2, x1:x2]
            det_scores.append(float(region.mean()) if region.size > 0 else 0.0)
        det_scores = np.array(det_scores)
        score_label = "depth"
    else:
        areas = (xyxy[:,2]-xyxy[:,0]) * (xyxy[:,3]-xyxy[:,1]) / (SW*SH)
        det_scores = confs * areas
        score_label = "conf*area"

    max_score = det_scores.max()
    fg_mask   = (det_scores >= max_score - DEPTH_TOLERANCE) & (confs >= CONF_THRESHOLD)
    if not fg_mask.any():
        fg_mask[np.argmax(det_scores)] = True

    print(f"\n{'idx':>3}  {'label':<15}  {'conf':>5}  {score_label:>10}  selected")
    print("-" * 52)
    fg_detections = []
    for i in range(len(yolo_dets)):
        label  = det.names[clses[i]]
        marker = "  [FG]" if fg_mask[i] else ""
        print(f"{i:>3}  {label:<15}  {confs[i]:.3f}  {det_scores[i]:>10.3f}{marker}")
        if fg_mask[i]:
            x1, y1, x2, y2 = xyxy[i]
            fg_detections.append({
                "label":    label,
                "box_norm": [x1/SW, y1/SH, (x2-x1)/SW, (y2-y1)/SH],
            })

print(f"\n{len(fg_detections)} foreground object(s) selected.")


**Build SAM3 video predictor**

In [ ]:
gpu_list = list(range(torch.cuda.device_count())) if device == "cuda" else []
print(f"Building SAM3 on {device}, GPUs={gpu_list}")

video_predictor = build_sam3_video_predictor(
    gpus_to_use=gpu_list if gpu_list else None
)
print("SAM3 ready.")


**Session + prompts + select objects**

In [ ]:
def iou_xywh(a, b):
    ax2, ay2 = a[0]+a[2], a[1]+a[3]
    bx2, by2 = b[0]+b[2], b[1]+b[3]
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
    ix2 = min(ax2,  bx2);  iy2 = min(ay2,  by2)
    inter = max(0.0, ix2-ix1) * max(0.0, iy2-iy1)
    union = a[2]*a[3] + b[2]*b[3] - inter
    return inter/union if union > 0 else 0.0

def mask_sum(m):
    arr = m.cpu().numpy() if isinstance(m, torch.Tensor) else np.asarray(m)
    return int(arr.sum())

with torch.inference_mode():
    # ── Start session ──────────────────────────────────────────────────────
    resp       = video_predictor.handle_request({
        "type":                 "start_session",
        "resource_path":        TEMP_VIDEO,
        "offload_video_to_cpu": True,
        "offload_state_to_cpu": True,
    })
    session_id = resp["session_id"]
    print(f"Session: {session_id}\n")

    # ── Text prompts — one per unique foreground class ─────────────────────
    all_obj_ids   = []
    all_sam_boxes = []
    all_sam_probs = []
    all_sam_masks = []

    prompt_labels = list({d["label"] for d in fg_detections}) if fg_detections else ["main subject"]

    for label in prompt_labels:
        print(f"Text prompt: '{label}'")
        r   = video_predictor.handle_request({
            "type": "add_prompt", "session_id": session_id,
            "frame_index": 0, "text": label,
        })
        out   = r.get("outputs", {})
        oids  = list(out.get("out_obj_ids",     []))
        boxes = list(out.get("out_boxes_xywh",  []))
        probs = list(out.get("out_probs",        []))
        masks = list(out.get("out_binary_masks", []))
        print(f"  -> {len(oids)} SAM3 object(s)")
        all_obj_ids   += oids
        all_sam_boxes += boxes
        all_sam_probs += probs
        all_sam_masks += masks

    # ── Select target obj_ids via IoU matching ─────────────────────────────
    target_obj_ids = set()

    if len(all_obj_ids) == 0:
        print("\nText prompts found nothing. Using multi-point fallback from YOLO box.")
        if fg_detections:
            xb, yb, wb, hb = fg_detections[0]["box_norm"]
        else:
            xb, yb, wb, hb = 0.25, 0.25, 0.50, 0.50
        fg_pts = [
            [xb+wb*0.50, yb+hb*0.50],
            [xb+wb*0.25, yb+hb*0.50],
            [xb+wb*0.75, yb+hb*0.50],
            [xb+wb*0.50, yb+hb*0.25],
            [xb+wb*0.50, yb+hb*0.75],
        ]
        bg_pts = [[0.02,0.02],[0.98,0.02],[0.02,0.98],[0.98,0.98]]
        pt_r = video_predictor.handle_request({
            "type": "add_prompt", "session_id": session_id,
            "frame_index": 0,
            "points": fg_pts + bg_pts,
            "point_labels": [1]*5 + [0]*4,
            "obj_id": 1,
        })
        target_obj_ids.add(1)
        # Multi-point prompts register the object directly — skip the step below
        all_obj_ids = []

    else:
        if fg_detections:
            print("\nIoU matching YOLO boxes to SAM3 objects:")
            for fd in fg_detections:
                ious     = [iou_xywh(fd["box_norm"], bx) for bx in all_sam_boxes]
                best_idx = int(np.argmax(ious))
                if ious[best_idx] < 0.02:
                    best_idx = int(np.argmax([mask_sum(m) for m in all_sam_masks]))
                    print(f"  '{fd['label']}': low IoU, largest mask -> obj_id={int(all_obj_ids[best_idx])}")
                else:
                    print(f"  '{fd['label']}': IoU={ious[best_idx]:.3f} -> obj_id={int(all_obj_ids[best_idx])}")
                target_obj_ids.add(int(all_obj_ids[best_idx]))
        else:
            best_idx = int(np.argmax([mask_sum(m) for m in all_sam_masks]))
            target_obj_ids.add(int(all_obj_ids[best_idx]))

    print(f"\nTracking obj_ids: {sorted(target_obj_ids)}")
    print("\nAll SAM3 objects on frame 0:")
    for oid, bx, pr, msk in zip(all_obj_ids, all_sam_boxes, all_sam_probs, all_sam_masks):
        sel = "  <-- tracked" if int(oid) in target_obj_ids else ""
        print(f"  obj_id={oid}  prob={pr:.3f}  mask_px={mask_sum(msk):>8,}{sel}")

    # ── CRITICAL: register each target for video tracking ─────────────────
    # Text-prompt detections are NOT automatically propagated by SAM3.
    # An explicit point prompt must be added for each target object to
    # register it as a video tracking target before propagate_in_video.
    if all_obj_ids:  # skip if we already used multi-point fallback above
        obj_id_to_box = {int(oid): box for oid, box in zip(all_obj_ids, all_sam_boxes)}
        print("\nRegistering targets for video tracking:")
        for tid in sorted(target_obj_ids):
            if tid not in obj_id_to_box:
                print(f"  obj_id={tid}: no box found, skipping")
                continue
            bx = obj_id_to_box[tid]
            cx = bx[0] + bx[2] / 2
            cy = bx[1] + bx[3] / 2
            video_predictor.handle_request({
                "type":         "add_prompt",
                "session_id":   session_id,
                "frame_index":  0,
                "points":       [[cx, cy]],
                "point_labels": [1],
                "obj_id":       tid,
            })
            print(f"  obj_id={tid}: registered at center ({cx:.3f}, {cy:.3f})")

    print("\nReady to propagate.")


**Propogate masks**

In [ ]:
all_masks  = {}
mask_count = 0

with torch.inference_mode():
    stream = video_predictor.handle_stream_request({
        "type":            "propagate_in_video",
        "session_id":      session_id,
        "score_threshold": 0.0,
    })
    for frame_data in tqdm(stream, total=n_frames, desc="Propagating"):
        fidx    = frame_data["frame_index"]
        outputs = frame_data.get("outputs")
        if not outputs:
            continue

        obj_ids_frame = outputs.get("out_obj_ids",      [])
        masks_frame   = outputs.get("out_binary_masks",  [])

        combined  = np.zeros((SH, SW), dtype=bool)
        found_any = False
        for oid, mask in zip(obj_ids_frame, masks_frame):
            if int(oid) not in target_obj_ids:
                continue
            if isinstance(mask, torch.Tensor):
                mask = mask.cpu().numpy()
            m = mask.squeeze().astype(bool)
            if m.any():
                combined  |= m
                found_any  = True

        if not found_any:
            continue

        mask_full = cv2.resize(combined.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
        mask_soft = cv2.GaussianBlur(mask_full.astype(np.float32), (7, 7), 0)
        all_masks[fidx] = mask_soft > 0.3
        mask_count += 1

print(f"Masks collected: {mask_count} / {n_frames} frames")
if mask_count == 0:
    print("ERROR: still 0 masks. Check Cell 7 registration output above.")


**Write output**

In [ ]:
cap       = cv2.VideoCapture(INPUT_VIDEO)
final_vid = cv2.VideoWriter(
    OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H)
)

for frame_idx in tqdm(range(n_frames), desc="Writing"):
    ret, frame = cap.read()
    if not ret:
        break
    mask = all_masks.get(frame_idx)
    if mask is not None:
        mask_uint8 = mask.astype(np.uint8) * 255

        # Gaussian blur on the mask
        blur_radius = 15
        blurred_mask = cv2.GaussianBlur(
            mask_uint8,
            (blur_radius, blur_radius),
            sigmaX=0
        )

        alpha = blurred_mask.astype(np.float32) / 255.0
        alpha = np.stack([alpha, alpha, alpha], axis=-1)

        bg = np.full_like(frame, BG_COLOR, dtype=np.uint8)
        frame = (frame * alpha + bg * (1.0 - alpha)).astype(np.uint8)

    final_vid.write(frame)

cap.release()
final_vid.release()

if os.path.exists(TEMP_VIDEO):
    os.remove(TEMP_VIDEO)

print(f"Done -> {OUTPUT_VIDEO}")